# 26 — Robustness, Domain Shift, and Ultrasound Harmonization in Depth

In the previous notebook, we learned how to design research-grade experiments using:

- Patient-grouped cross-validation
- Stratified folds
- Multiple random seeds
- Leakage-safe hyperparameter tuning
- Ablation studies
- Patient-level bootstrap
- External validation

Now we will focus on one of the hardest problems in medical imaging:

> **A model can perform very well on the development dataset and still fail when the scanner, hospital, acquisition protocol, or image statistics change.**

This is called:

> **Domain shift**

For ultrasound, domain shift is especially important because appearance can change because of:

- Scanner manufacturer
- Probe type
- Gain
- Dynamic range
- Depth
- Frequency
- Post-processing
- Operator technique
- Site-specific protocols

## In this notebook, we will study:

1. What is domain shift?
2. Scanner and site shift
3. Covariate shift
4. Label shift intuition
5. Measuring distribution differences
6. Site-held-out evaluation
7. Device-held-out evaluation
8. Robustness stress tests
9. Intensity perturbation tests
10. Resolution and noise sensitivity
11. Harmonization goals
12. Global vs per-image normalization
13. Histogram-based harmonization
14. Feature-space harmonization intuition
15. Domain-invariant representation learning
16. Detecting shortcut learning across sites
17. Evaluating whether harmonization truly improves generalization
18. Common harmonization mistakes
19. Practice exercises

## Main Goal

The robustness question is:

$$
\boxed{
\text{Does the model learn clinically useful signal}
\quad
\text{or domain-specific shortcuts?}
}
$$

A strong harmonization method should ideally:

$$
\boxed{
\text{Reduce nuisance variation}
+
\text{Preserve label information}
+
\text{Improve unseen-domain generalization}
}
$$

The most important principle is:

> **A harmonization method is useful only if it improves the behavior of the downstream task under realistic domain shift.**


In [ ]:
import copy
import json
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

print("PyTorch:", torch.__version__)


# 1. What Is Domain Shift?

Suppose training data come from distribution:

$$
P_{train}(X,Y)
$$

but deployment data come from:

$$
P_{test}(X,Y)
$$

If:

$$
\boxed{
P_{train}(X,Y)
\neq
P_{test}(X,Y)
}
$$

we have some form of **distribution shift**.

In medical imaging, this can happen even when the clinical task stays exactly the same.


# 2. Ultrasound Domain Shift

Two ultrasound images of the same anatomy can look very different because of:

$$
\begin{array}{|c|c|}
\hline
\textbf{Source of Shift} & \textbf{Possible Effect} \\
\hline
Scanner & Contrast / post-processing \\
\hline
Probe & Resolution / frequency response \\
\hline
Gain & Brightness distribution \\
\hline
Depth & Scale / field of view \\
\hline
Site & Protocol and workflow \\
\hline
Operator & View / orientation / pressure \\
\hline
Software version & Image rendering differences \\
\hline
\end{array}
$$


# 3. Scanner Shift

Scanner shift means:

> The image distribution changes because acquisition hardware or image processing changes.

Examples:

- Device A produces darker images
- Device B applies stronger edge enhancement
- Device C has different speckle statistics

A model may learn these differences if they correlate with labels.


# 4. Site Shift

Site shift can include more than scanner changes.

Different hospitals may differ in:

- Patient population
- Scanner fleet
- Probe usage
- Operator expertise
- Referral patterns
- Disease prevalence
- Reporting conventions

Therefore:

$$
Site\ Shift
$$

can combine:

$$
Image\ Shift
+
Population\ Shift
+
Workflow\ Shift
$$


# 5. Covariate Shift

A simplified form of domain shift is:

$$
\boxed{
P_{train}(X)
\neq
P_{test}(X)
}
$$

while the relationship between image and label is assumed similar.

This is called:

> **Covariate shift**

Example:

- Scanner changes image contrast
- Disease definition stays the same


# 6. Label Shift Intuition

Label shift means the class prevalence changes:

$$
\boxed{
P_{train}(Y)
\neq
P_{test}(Y)
}
$$

Example:

Training cohort:

$$
50\%
$$

disease prevalence.

Deployment hospital:

$$
10\%
$$

disease prevalence.

This can strongly affect:

- Precision
- Calibration
- Decision thresholds


# 7. Concept Shift

A more difficult problem is when:

$$
P(Y|X)
$$

itself changes.

This can happen because of:

- Different labeling criteria
- Different disease definitions
- Different annotation quality
- Different clinical population

No simple intensity harmonization can solve a true labeling/concept mismatch.


# 8. Domain Shift Categories

A useful mental model:

$$
\begin{array}{|c|c|}
\hline
\textbf{Shift Type} & \textbf{What Changes?} \\
\hline
Covariate\ shift & P(X) \\
\hline
Label\ shift & P(Y) \\
\hline
Concept\ shift & P(Y|X) \\
\hline
\end{array}
$$


# 9. Build a Synthetic Multi-Domain Ultrasound Dataset

To study the ideas directly, we will create a synthetic dataset with:

- Three classes
- Three development sites
- Three devices
- One unseen external site/device
- Site/device-specific intensity transformations

The task signal will stay the same.

Only the image domain changes.


In [ ]:
def make_base_ultrasound_pattern(
    class_index,
    image_size=48
):
    image = torch.zeros(
        1,
        image_size,
        image_size
    )

    center = image_size // 2

    if class_index == 0:
        image[
            :,
            6:image_size - 6,
            center - 2:center + 3
        ] = 0.85

    elif class_index == 1:
        image[
            :,
            center - 2:center + 3,
            6:image_size - 6
        ] = 0.85

    elif class_index == 2:
        image[
            :,
            center - 7:center + 8,
            center - 7:center + 8
        ] = 0.85

    else:
        raise ValueError(
            "class_index must be 0, 1, or 2"
        )

    return image


# 10. Domain Transformation Function

Each site/device will change:

- Gain
- Offset
- Contrast
- Blur strength
- Noise level

This creates controlled covariate shift.


In [ ]:
def blur_image(
    image,
    kernel_size=3
):
    if kernel_size == 1:
        return image

    padding = (
        kernel_size
        // 2
    )

    return F.avg_pool2d(
        image.unsqueeze(0),
        kernel_size=kernel_size,
        stride=1,
        padding=padding
    ).squeeze(0)


def apply_domain_transform(
    image,
    site,
    device,
    generator
):
    site_gain = {
        "Site_A": 0.85,
        "Site_B": 1.00,
        "Site_C": 1.12,
        "External_Site": 0.72,
    }[
        site
    ]

    site_offset = {
        "Site_A": 0.03,
        "Site_B": 0.00,
        "Site_C": 0.06,
        "External_Site": 0.10,
    }[
        site
    ]

    device_contrast = {
        "Device_A": 0.90,
        "Device_B": 1.00,
        "Device_C": 1.15,
        "Device_X": 1.28,
    }[
        device
    ]

    blur_kernel = {
        "Device_A": 1,
        "Device_B": 3,
        "Device_C": 3,
        "Device_X": 5,
    }[
        device
    ]

    noise_std = {
        "Device_A": 0.06,
        "Device_B": 0.09,
        "Device_C": 0.12,
        "Device_X": 0.16,
    }[
        device
    ]

    image = (
        image
        * site_gain
        + site_offset
    )

    image = (
        image
        - 0.5
    ) * device_contrast + 0.5

    image = blur_image(
        image,
        kernel_size=blur_kernel
    )

    noise = torch.randn(
        image.shape,
        generator=generator
    ) * noise_std

    image = (
        image
        + noise
    ).clamp(
        0.0,
        1.0
    )

    return image


# 11. Add Patient-Level Variation

Each patient contributes multiple correlated images.

That keeps the experimental unit realistic.


In [ ]:
def create_multidomain_dataset(
    num_internal_patients=90,
    num_external_patients=24,
    images_per_patient=2,
    image_size=48
):
    images = []
    records = []

    internal_sites = [
        "Site_A",
        "Site_B",
        "Site_C"
    ]

    internal_devices = [
        "Device_A",
        "Device_B",
        "Device_C"
    ]

    for patient_index in range(
        num_internal_patients
    ):
        patient_id = (
            f"I{patient_index:03d}"
        )

        label = (
            patient_index
            % 3
        )

        site = internal_sites[
            patient_index
            % len(
                internal_sites
            )
        ]

        device = internal_devices[
            (
                patient_index
                + patient_index // 3
            )
            % len(
                internal_devices
            )
        ]

        generator = (
            torch.Generator()
            .manual_seed(
                1000
                + patient_index
            )
        )

        for image_index in range(
            images_per_patient
        ):
            base = (
                make_base_ultrasound_pattern(
                    label,
                    image_size=image_size
                )
            )

            shift_y = int(
                torch.randint(
                    -4,
                    5,
                    (1,),
                    generator=generator
                ).item()
            )

            shift_x = int(
                torch.randint(
                    -4,
                    5,
                    (1,),
                    generator=generator
                ).item()
            )

            base = torch.roll(
                base,
                shifts=(
                    shift_y,
                    shift_x
                ),
                dims=(
                    1,
                    2
                )
            )

            image = (
                apply_domain_transform(
                    base,
                    site,
                    device,
                    generator
                )
            )

            tensor_index = len(
                images
            )

            images.append(
                image
            )

            records.append({
                "tensor_index":
                    tensor_index,

                "patient_id":
                    patient_id,

                "label":
                    label,

                "site":
                    site,

                "device":
                    device,

                "cohort":
                    "internal",

                "image_index":
                    image_index
            })

    for patient_index in range(
        num_external_patients
    ):
        patient_id = (
            f"E{patient_index:03d}"
        )

        label = (
            patient_index
            % 3
        )

        site = (
            "External_Site"
        )

        device = (
            "Device_X"
        )

        generator = (
            torch.Generator()
            .manual_seed(
                5000
                + patient_index
            )
        )

        for image_index in range(
            images_per_patient
        ):
            base = (
                make_base_ultrasound_pattern(
                    label,
                    image_size=image_size
                )
            )

            shift_y = int(
                torch.randint(
                    -4,
                    5,
                    (1,),
                    generator=generator
                ).item()
            )

            shift_x = int(
                torch.randint(
                    -4,
                    5,
                    (1,),
                    generator=generator
                ).item()
            )

            base = torch.roll(
                base,
                shifts=(
                    shift_y,
                    shift_x
                ),
                dims=(
                    1,
                    2
                )
            )

            image = (
                apply_domain_transform(
                    base,
                    site,
                    device,
                    generator
                )
            )

            tensor_index = len(
                images
            )

            images.append(
                image
            )

            records.append({
                "tensor_index":
                    tensor_index,

                "patient_id":
                    patient_id,

                "label":
                    label,

                "site":
                    site,

                "device":
                    device,

                "cohort":
                    "external",

                "image_index":
                    image_index
            })

    return (
        images,
        pd.DataFrame(
            records
        )
    )


domain_images, domain_metadata = (
    create_multidomain_dataset()
)

print(
    "Images:",
    len(
        domain_images
    )
)

print(
    "Patients:",
    domain_metadata[
        "patient_id"
    ].nunique()
)


# 12. Inspect Site and Device Counts


In [ ]:
print(
    pd.crosstab(
        domain_metadata[
            "site"
        ],
        domain_metadata[
            "label"
        ]
    )
)

print()

print(
    pd.crosstab(
        domain_metadata[
            "device"
        ],
        domain_metadata[
            "label"
        ]
    )
)


# 13. Visualize One Image From Each Domain


In [ ]:
display_domains = [
    ("Site_A", "Device_A"),
    ("Site_B", "Device_B"),
    ("Site_C", "Device_C"),
    ("External_Site", "Device_X"),
]

fig, axes = plt.subplots(
    1,
    4,
    figsize=(12, 3)
)

for axis, (
    site,
    device
) in zip(
    axes,
    display_domains
):
    row = domain_metadata[
        (
            domain_metadata[
                "site"
            ]
            == site
        )
        &
        (
            domain_metadata[
                "device"
            ]
            == device
        )
    ].iloc[
        0
    ]

    image = domain_images[
        int(
            row[
                "tensor_index"
            ]
        )
    ]

    axis.imshow(
        image.squeeze(0).numpy(),
        cmap="gray"
    )

    axis.set_title(
        f"{site}\n{device}"
    )

    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 14. Measuring Distribution Differences

Before training a model, compare domains using simple statistics such as:

- Mean intensity
- Standard deviation
- Quantiles
- Histograms
- Image entropy
- Feature distributions

No single statistic captures all forms of shift.


In [ ]:
def image_statistics(
    image
):
    flat = image.flatten()

    return {
        "mean":
            float(
                flat.mean().item()
            ),

        "std":
            float(
                flat.std().item()
            ),

        "q10":
            float(
                torch.quantile(
                    flat,
                    0.10
                ).item()
            ),

        "q50":
            float(
                torch.quantile(
                    flat,
                    0.50
                ).item()
            ),

        "q90":
            float(
                torch.quantile(
                    flat,
                    0.90
                ).item()
            )
    }


# 15. Domain-Level Intensity Statistics


In [ ]:
domain_stat_rows = []

for site, group in (
    domain_metadata.groupby(
        "site"
    )
):
    site_images = [
        domain_images[
            int(
                idx
            )
        ]
        for idx in group[
            "tensor_index"
        ].tolist()
    ]

    stacked = torch.stack(
        site_images
    )

    domain_stat_rows.append({
        "site":
            site,

        "mean":
            float(
                stacked.mean().item()
            ),

        "std":
            float(
                stacked.std().item()
            ),

        "q10":
            float(
                torch.quantile(
                    stacked.flatten(),
                    0.10
                ).item()
            ),

        "q50":
            float(
                torch.quantile(
                    stacked.flatten(),
                    0.50
                ).item()
            ),

        "q90":
            float(
                torch.quantile(
                    stacked.flatten(),
                    0.90
                ).item()
            )
    })

domain_stats_df = pd.DataFrame(
    domain_stat_rows
)

print(
    domain_stats_df
)


# 16. Histogram Comparison

Histograms give a richer picture than mean/std alone.

We will compare grayscale distributions across sites.


In [ ]:
plt.figure(figsize=(8, 5))

for site in domain_metadata[
    "site"
].unique():
    indices = domain_metadata[
        domain_metadata[
            "site"
        ]
        == site
    ][
        "tensor_index"
    ].tolist()

    values = torch.cat([
        domain_images[
            int(
                index
            )
        ].flatten()
        for index in indices
    ])

    hist = torch.histc(
        values,
        bins=50,
        min=0.0,
        max=1.0
    )

    hist = (
        hist
        / hist.sum()
    )

    centers = torch.linspace(
        0.01,
        0.99,
        50
    )

    plt.plot(
        centers.numpy(),
        hist.numpy(),
        label=site
    )

plt.xlabel(
    "Intensity"
)

plt.ylabel(
    "Normalized Frequency"
)

plt.title(
    "Site Intensity Histograms"
)

plt.legend()
plt.show()


# 17. Simple Distribution Distance — Mean Difference

One very crude distance is:

$$
\boxed{
|\mu_A-\mu_B|
}
$$

This only measures one aspect of the domain gap.


In [ ]:
site_means = {
    row[
        "site"
    ]:
        row[
            "mean"
        ]
    for _, row in domain_stats_df.iterrows()
}

for site_a in site_means:
    for site_b in site_means:
        if site_a < site_b:
            print(
                site_a,
                site_b,
                abs(
                    site_means[
                        site_a
                    ]
                    - site_means[
                        site_b
                    ]
                )
            )


# 18. Maximum Mean Discrepancy Intuition

A more flexible distribution-distance idea is:

> **Maximum Mean Discrepancy — MMD**

MMD compares distributions in a feature/kernel space.

For teaching, we will implement a small Gaussian-kernel version on flattened image features.


In [ ]:
def rbf_kernel(
    x,
    y,
    gamma
):
    x_sq = (
        x.pow(
            2
        ).sum(
            dim=1,
            keepdim=True
        )
    )

    y_sq = (
        y.pow(
            2
        ).sum(
            dim=1,
            keepdim=True
        ).T
    )

    distances = (
        x_sq
        + y_sq
        - 2.0
        * x
        @ y.T
    ).clamp_min(
        0.0
    )

    return torch.exp(
        -gamma
        * distances
    )


def mmd_rbf(
    x,
    y,
    gamma=0.01
):
    k_xx = rbf_kernel(
        x,
        x,
        gamma
    ).mean()

    k_yy = rbf_kernel(
        y,
        y,
        gamma
    ).mean()

    k_xy = rbf_kernel(
        x,
        y,
        gamma
    ).mean()

    return float(
        (
            k_xx
            + k_yy
            - 2.0
            * k_xy
        ).item()
    )


# 19. Compute MMD Between Sites

To keep computation light, we downsample flattened images.


In [ ]:
def site_feature_matrix(
    metadata,
    images,
    site,
    max_images=24
):
    indices = metadata[
        metadata[
            "site"
        ]
        == site
    ][
        "tensor_index"
    ].tolist()[
        :max_images
    ]

    features = torch.stack([
        F.adaptive_avg_pool2d(
            images[
                int(
                    index
                )
            ].unsqueeze(
                0
            ),
            output_size=(
                12,
                12
            )
        ).flatten()
        for index in indices
    ])

    return features


site_a_features = site_feature_matrix(
    domain_metadata,
    domain_images,
    "Site_A"
)

external_features = site_feature_matrix(
    domain_metadata,
    domain_images,
    "External_Site"
)

print(
    "MMD:",
    mmd_rbf(
        site_a_features,
        external_features,
        gamma=0.05
    )
)


# 20. Distribution Difference Does Not Equal Performance Difference

A large image-distribution gap does not automatically mean large classification failure.

A domain change may affect:

- Background
- Intensity
- Borders

while leaving the task signal intact.

Therefore always connect distribution measurements to downstream performance.


# 21. Dataset Class With Harmonization Modes

We will compare:

- `none`
- `global`
- `per_image`
- `histogram_reference`

Every learned reference must come from training data only.


In [ ]:
def compute_global_mean_std(
    metadata,
    images
):
    stacked = torch.stack([
        images[
            int(
                index
            )
        ]
        for index in metadata[
            "tensor_index"
        ].tolist()
    ])

    return (
        float(
            stacked.mean().item()
        ),
        float(
            stacked.std().item()
        )
    )


# 22. Histogram Reference

For histogram-based harmonization, we can create a reference cumulative distribution from the training images.

This is a simplified educational implementation.


In [ ]:
def build_histogram_reference(
    metadata,
    images,
    bins=128
):
    all_values = torch.cat([
        images[
            int(
                index
            )
        ].flatten()
        for index in metadata[
            "tensor_index"
        ].tolist()
    ])

    hist = torch.histc(
        all_values,
        bins=bins,
        min=0.0,
        max=1.0
    )

    hist = (
        hist
        / hist.sum()
    )

    cdf = torch.cumsum(
        hist,
        dim=0
    )

    bin_centers = torch.linspace(
        0.0,
        1.0,
        bins
    )

    return {
        "hist":
            hist,

        "cdf":
            cdf,

        "bin_centers":
            bin_centers
    }


# 23. Histogram Matching Intuition

Histogram matching tries to transform an image so its cumulative intensity distribution resembles a reference.

Conceptually:

$$
x
\rightarrow
CDF_{source}(x)
\rightarrow
CDF^{-1}_{reference}
$$

This changes global intensity statistics.

It does not directly align anatomy or spatial texture.


In [ ]:
def histogram_match_tensor(
    image,
    reference,
    bins=128
):
    flat = image.flatten()

    source_hist = torch.histc(
        flat,
        bins=bins,
        min=0.0,
        max=1.0
    )

    source_hist = (
        source_hist
        / source_hist.sum()
    )

    source_cdf = torch.cumsum(
        source_hist,
        dim=0
    )

    ref_cdf = reference[
        "cdf"
    ]

    ref_centers = reference[
        "bin_centers"
    ]

    source_bins = torch.clamp(
        (
            flat
            * (
                bins - 1
            )
        ).long(),
        0,
        bins - 1
    )

    source_quantiles = source_cdf[
        source_bins
    ]

    mapped_indices = torch.searchsorted(
        ref_cdf,
        source_quantiles
    )

    mapped_indices = mapped_indices.clamp(
        0,
        bins - 1
    )

    matched = ref_centers[
        mapped_indices
    ].view_as(
        image
    )

    return matched


# 24. Visualize Histogram Matching


In [ ]:
internal_only = domain_metadata[
    domain_metadata[
        "cohort"
    ]
    == "internal"
].reset_index(
    drop=True
)

global_reference = (
    build_histogram_reference(
        internal_only,
        domain_images
    )
)

external_row = domain_metadata[
    domain_metadata[
        "site"
    ]
    == "External_Site"
].iloc[
    0
]

external_image = domain_images[
    int(
        external_row[
            "tensor_index"
        ]
    )
]

matched_image = histogram_match_tensor(
    external_image,
    global_reference
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7, 3)
)

axes[0].imshow(
    external_image.squeeze(0).numpy(),
    cmap="gray"
)

axes[0].set_title(
    "External original"
)

axes[1].imshow(
    matched_image.squeeze(0).numpy(),
    cmap="gray"
)

axes[1].set_title(
    "Histogram matched"
)

for axis in axes:
    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 25. Why Histogram Matching Can Help

It may reduce:

- Brightness shift
- Contrast shift
- Global intensity-distribution differences

But it may not solve:

- Resolution shift
- Noise differences
- Spatial artifacts
- Probe/view differences
- Label shift


# 26. Why Histogram Matching Can Hurt

If clinically relevant information depends on intensity patterns, aggressive matching can remove useful signal.

This is especially important in ultrasound where echogenicity itself may carry meaning.


In [ ]:
class DomainDataset(Dataset):
    def __init__(
        self,
        metadata,
        images,
        training,
        harmonization,
        train_mean=None,
        train_std=None,
        histogram_reference=None
    ):
        self.metadata = (
            metadata
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.images = images
        self.training = training
        self.harmonization = harmonization
        self.train_mean = train_mean
        self.train_std = train_std
        self.histogram_reference = (
            histogram_reference
        )

    def __len__(self):
        return len(
            self.metadata
        )

    def __getitem__(
        self,
        index
    ):
        row = self.metadata.iloc[
            index
        ]

        image = self.images[
            int(
                row[
                    "tensor_index"
                ]
            )
        ].clone()

        if self.training:
            shift = int(
                torch.randint(
                    -2,
                    3,
                    (1,)
                ).item()
            )

            image = torch.roll(
                image,
                shifts=shift,
                dims=2
            )

        if self.harmonization == "none":
            pass

        elif self.harmonization == "global":
            image = (
                image
                - self.train_mean
            ) / (
                self.train_std
                + 1e-8
            )

        elif self.harmonization == "per_image":
            image = (
                image
                - image.mean()
            ) / (
                image.std()
                + 1e-8
            )

        elif self.harmonization == "histogram_reference":
            image = histogram_match_tensor(
                image,
                self.histogram_reference
            )

        else:
            raise ValueError(
                "Unknown harmonization mode."
            )

        target = torch.tensor(
            int(
                row[
                    "label"
                ]
            ),
            dtype=torch.long
        )

        return (
            image,
            target,
            index
        )


# 27. Baseline CNN


In [ ]:
class RobustnessCNN(nn.Module):
    def __init__(
        self,
        num_classes=3
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                16,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool2d(
            1
        )

        self.classifier = nn.Linear(
            64,
            num_classes
        )

    def forward(
        self,
        x
    ):
        x = self.features(
            x
        )

        x = self.pool(
            x
        )

        x = torch.flatten(
            x,
            start_dim=1
        )

        return self.classifier(
            x
        )


# 28. Device Setup


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)


# 29. Training Helper


In [ ]:
def set_seed(
    seed
):
    random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for images, targets, _ in loader:
        images = images.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()
        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += (
            batch_size
        )

    return (
        total_loss
        / total_samples
    )


# 30. Evaluation Helper


In [ ]:
def evaluate_model(
    model,
    dataset,
    loader,
    device
):
    model.eval()

    rows = []

    with torch.inference_mode():
        for images, targets, indices in loader:
            images = images.to(
                device
            )

            logits = model(
                images
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            ).cpu()

            predictions = logits.argmax(
                dim=1
            ).cpu()

            for position in range(
                len(
                    indices
                )
            ):
                local_index = int(
                    indices[
                        position
                    ].item()
                )

                metadata_row = (
                    dataset
                    .metadata
                    .iloc[
                        local_index
                    ]
                )

                record = {
                    "patient_id":
                        metadata_row[
                            "patient_id"
                        ],

                    "site":
                        metadata_row[
                            "site"
                        ],

                    "device":
                        metadata_row[
                            "device"
                        ],

                    "true_label":
                        int(
                            targets[
                                position
                            ].item()
                        ),

                    "predicted_label":
                        int(
                            predictions[
                                position
                            ].item()
                        )
                }

                for class_index in range(
                    probabilities.shape[
                        1
                    ]
                ):
                    record[
                        f"prob_class_{class_index}"
                    ] = float(
                        probabilities[
                            position,
                            class_index
                        ].item()
                    )

                rows.append(
                    record
                )

    dataframe = pd.DataFrame(
        rows
    )

    accuracy = (
        dataframe[
            "true_label"
        ].to_numpy()
        ==
        dataframe[
            "predicted_label"
        ].to_numpy()
    ).mean()

    return (
        float(
            accuracy
        ),
        dataframe
    )


# 31. Standard Internal Split for Demonstrations

We will create:

- Internal training patients
- Internal validation patients
- External test patients

The external site stays untouched during development.


In [ ]:
internal_patient_table = (
    internal_only
    .groupby(
        "patient_id",
        as_index=False
    )
    .agg({
        "label":
            "first",

        "site":
            "first",

        "device":
            "first"
    })
)

train_patient_ids = []
val_patient_ids = []

for class_index in sorted(
    internal_patient_table[
        "label"
    ].unique()
):
    class_ids = (
        internal_patient_table[
            internal_patient_table[
                "label"
            ]
            == class_index
        ][
            "patient_id"
        ]
        .tolist()
    )

    random.Random(
        42
        + int(
            class_index
        )
    ).shuffle(
        class_ids
    )

    split_point = int(
        0.8
        * len(
            class_ids
        )
    )

    train_patient_ids.extend(
        class_ids[
            :split_point
        ]
    )

    val_patient_ids.extend(
        class_ids[
            split_point:
        ]
    )

train_patient_ids = set(
    train_patient_ids
)

val_patient_ids = set(
    val_patient_ids
)

print(
    len(train_patient_ids),
    len(val_patient_ids)
)


# 32. Internal Train / Validation Metadata


In [ ]:
train_metadata = (
    internal_only[
        internal_only[
            "patient_id"
        ].isin(
            train_patient_ids
        )
    ]
    .reset_index(
        drop=True
    )
)

val_metadata = (
    internal_only[
        internal_only[
            "patient_id"
        ].isin(
            val_patient_ids
        )
    ]
    .reset_index(
        drop=True
    )
)

external_metadata = (
    domain_metadata[
        domain_metadata[
            "cohort"
        ]
        == "external"
    ]
    .reset_index(
        drop=True
    )
)

print(
    len(train_metadata),
    len(val_metadata),
    len(external_metadata)
)


# 33. Fit Harmonization Only on Training Data

This is essential.

We compute:

- Global mean/std
- Histogram reference

using **training data only**.


In [ ]:
train_mean, train_std = (
    compute_global_mean_std(
        train_metadata,
        domain_images
    )
)

train_hist_reference = (
    build_histogram_reference(
        train_metadata,
        domain_images
    )
)

print(
    train_mean,
    train_std
)


# 34. Train One Harmonization Variant


In [ ]:
def train_harmonization_variant(
    mode,
    train_metadata,
    val_metadata,
    images,
    train_mean,
    train_std,
    histogram_reference,
    seed=42,
    epochs=4
):
    set_seed(
        seed
    )

    train_dataset = DomainDataset(
        train_metadata,
        images,
        training=True,
        harmonization=mode,
        train_mean=train_mean,
        train_std=train_std,
        histogram_reference=(
            histogram_reference
        )
    )

    val_dataset = DomainDataset(
        val_metadata,
        images,
        training=False,
        harmonization=mode,
        train_mean=train_mean,
        train_std=train_std,
        histogram_reference=(
            histogram_reference
        )
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=24,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=48,
        shuffle=False,
        num_workers=0
    )

    model = RobustnessCNN(
        num_classes=3
    ).to(
        device
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    best_accuracy = -1.0
    best_state = copy.deepcopy(
        model.state_dict()
    )

    for _ in range(
        epochs
    ):
        train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_accuracy, _ = evaluate_model(
            model,
            val_dataset,
            val_loader,
            device
        )

        if val_accuracy > best_accuracy:
            best_accuracy = (
                val_accuracy
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

    model.load_state_dict(
        best_state
    )

    final_accuracy, predictions = (
        evaluate_model(
            model,
            val_dataset,
            val_loader,
            device
        )
    )

    return {
        "model":
            model,

        "val_accuracy":
            final_accuracy,

        "predictions":
            predictions
    }


# 35. Compare Harmonization Variants on Internal Validation

We compare:

- No harmonization
- Global standardization
- Per-image standardization
- Histogram matching

Same:

- Train/validation patients
- Architecture
- Optimizer
- Epochs
- Seed


In [ ]:
harmonization_modes = [
    "none",
    "global",
    "per_image",
    "histogram_reference"
]

harmonization_results = {}

for mode in harmonization_modes:
    print(
        "Running:",
        mode
    )

    harmonization_results[
        mode
    ] = (
        train_harmonization_variant(
            mode,
            train_metadata,
            val_metadata,
            domain_images,
            train_mean,
            train_std,
            train_hist_reference,
            seed=42,
            epochs=3
        )
    )

for mode in harmonization_modes:
    print(
        mode,
        harmonization_results[
            mode
        ][
            "val_accuracy"
        ]
    )


# 36. Why Internal Validation Is Not Enough

A harmonization method may improve internal validation because it makes development sites look more similar.

But the real question is:

> Does it improve performance on a truly unseen domain?

So we must evaluate external generalization.


# 37. External Evaluation Function


In [ ]:
def evaluate_external(
    model,
    mode,
    external_metadata,
    images,
    train_mean,
    train_std,
    histogram_reference
):
    external_dataset = DomainDataset(
        external_metadata,
        images,
        training=False,
        harmonization=mode,
        train_mean=train_mean,
        train_std=train_std,
        histogram_reference=(
            histogram_reference
        )
    )

    external_loader = DataLoader(
        external_dataset,
        batch_size=48,
        shuffle=False,
        num_workers=0
    )

    return evaluate_model(
        model,
        external_dataset,
        external_loader,
        device
    )


# 38. External Accuracy by Harmonization Method


In [ ]:
external_rows = []

for mode in harmonization_modes:
    model = harmonization_results[
        mode
    ][
        "model"
    ]

    external_accuracy, external_predictions = (
        evaluate_external(
            model,
            mode,
            external_metadata,
            domain_images,
            train_mean,
            train_std,
            train_hist_reference
        )
    )

    external_rows.append({
        "harmonization":
            mode,

        "internal_val_accuracy":
            harmonization_results[
                mode
            ][
                "val_accuracy"
            ],

        "external_accuracy":
            external_accuracy
    })

external_comparison = pd.DataFrame(
    external_rows
)

print(
    external_comparison
)


# 39. The Correct Harmonization Question

Do not ask only:

> Which method has the highest internal validation score?

Ask:

$$
\boxed{
\text{Which method improves unseen-domain performance}
}
$$

without damaging clinically meaningful class information.


# 40. Site-Held-Out Evaluation

A strong domain-generalization test is:

> Train on all but one site, evaluate on the held-out site.

Example:

- Train: Site A + Site B
- Test: Site C

Repeat for every development site.


In [ ]:
def site_held_out_split(
    metadata,
    held_out_site
):
    train_meta = (
        metadata[
            metadata[
                "site"
            ]
            != held_out_site
        ]
        .reset_index(
            drop=True
        )
    )

    test_meta = (
        metadata[
            metadata[
                "site"
            ]
            == held_out_site
        ]
        .reset_index(
            drop=True
        )
    )

    return (
        train_meta,
        test_meta
    )


# 41. Site-Held-Out Experiment

Every preprocessing/harmonization statistic must be fitted on the non-held-out sites.


In [ ]:
def run_site_held_out(
    metadata,
    images,
    held_out_site,
    harmonization="global",
    seed=42,
    epochs=3
):
    train_meta, test_meta = (
        site_held_out_split(
            metadata,
            held_out_site
        )
    )

    train_mean, train_std = (
        compute_global_mean_std(
            train_meta,
            images
        )
    )

    reference = (
        build_histogram_reference(
            train_meta,
            images
        )
    )

    result = train_harmonization_variant(
        harmonization,
        train_meta,
        test_meta,
        images,
        train_mean,
        train_std,
        reference,
        seed=seed,
        epochs=epochs
    )

    return {
        "held_out_site":
            held_out_site,

        "accuracy":
            result[
                "val_accuracy"
            ]
    }


# 42. Run Site-Held-Out Evaluation


In [ ]:
site_holdout_results = []

for site in [
    "Site_A",
    "Site_B",
    "Site_C"
]:
    result = run_site_held_out(
        internal_only,
        domain_images,
        held_out_site=site,
        harmonization="global",
        seed=42,
        epochs=2
    )

    site_holdout_results.append(
        result
    )

site_holdout_df = pd.DataFrame(
    site_holdout_results
)

print(
    site_holdout_df
)


# 43. Device-Held-Out Evaluation

Exactly the same idea can be applied to devices.

Train on:

- Device A
- Device B

Test on:

- Device C

This tests robustness to scanner hardware/processing differences.


In [ ]:
def device_held_out_split(
    metadata,
    held_out_device
):
    train_meta = (
        metadata[
            metadata[
                "device"
            ]
            != held_out_device
        ]
        .reset_index(
            drop=True
        )
    )

    test_meta = (
        metadata[
            metadata[
                "device"
            ]
            == held_out_device
        ]
        .reset_index(
            drop=True
        )
    )

    return (
        train_meta,
        test_meta
    )


# 44. Device-Held-Out Experiment


In [ ]:
def run_device_held_out(
    metadata,
    images,
    held_out_device,
    harmonization="global",
    seed=42,
    epochs=3
):
    train_meta, test_meta = (
        device_held_out_split(
            metadata,
            held_out_device
        )
    )

    train_mean, train_std = (
        compute_global_mean_std(
            train_meta,
            images
        )
    )

    reference = (
        build_histogram_reference(
            train_meta,
            images
        )
    )

    result = train_harmonization_variant(
        harmonization,
        train_meta,
        test_meta,
        images,
        train_mean,
        train_std,
        reference,
        seed=seed,
        epochs=epochs
    )

    return {
        "held_out_device":
            held_out_device,

        "accuracy":
            result[
                "val_accuracy"
            ]
    }


# 45. Run Device-Held-Out Evaluation


In [ ]:
device_holdout_results = []

for device_name in [
    "Device_A",
    "Device_B",
    "Device_C"
]:
    result = run_device_held_out(
        internal_only,
        domain_images,
        held_out_device=(
            device_name
        ),
        harmonization="global",
        seed=42,
        epochs=2
    )

    device_holdout_results.append(
        result
    )

device_holdout_df = pd.DataFrame(
    device_holdout_results
)

print(
    device_holdout_df
)


# 46. Robustness Stress Tests

A robustness stress test intentionally perturbs inputs and measures performance degradation.

Examples:

- Brightness change
- Contrast change
- Added noise
- Resolution loss
- Blur
- Compression

These tests approximate plausible deployment shifts.


# 47. Intensity Gain Perturbation


In [ ]:
def perturb_gain(
    image,
    factor
):
    return (
        image
        * factor
    ).clamp(
        0.0,
        1.0
    )


# 48. Intensity Offset Perturbation


In [ ]:
def perturb_offset(
    image,
    offset
):
    return (
        image
        + offset
    ).clamp(
        0.0,
        1.0
    )


# 49. Contrast Perturbation


In [ ]:
def perturb_contrast(
    image,
    factor
):
    mean = image.mean()

    return (
        (
            image
            - mean
        )
        * factor
        + mean
    ).clamp(
        0.0,
        1.0
    )


# 50. Gaussian Noise Perturbation


In [ ]:
def perturb_noise(
    image,
    std,
    generator=None
):
    if generator is None:
        noise = torch.randn_like(
            image
        ) * std

    else:
        noise = torch.randn(
            image.shape,
            generator=generator
        ) * std

    return (
        image
        + noise
    ).clamp(
        0.0,
        1.0
    )


# 51. Resolution Degradation

Simulate lower resolution:

1. Downsample
2. Upsample back to original size

This removes high-frequency detail.


In [ ]:
def perturb_resolution(
    image,
    scale
):
    original_size = image.shape[
        -2:
    ]

    low_h = max(
        4,
        int(
            original_size[
                0
            ]
            * scale
        )
    )

    low_w = max(
        4,
        int(
            original_size[
                1
            ]
            * scale
        )
    )

    low = F.interpolate(
        image.unsqueeze(
            0
        ),
        size=(
            low_h,
            low_w
        ),
        mode="bilinear",
        align_corners=False
    )

    restored = F.interpolate(
        low,
        size=original_size,
        mode="bilinear",
        align_corners=False
    )

    return restored.squeeze(
        0
    )


# 52. Perturbation Dataset Wrapper

This applies a stress-test transform before harmonization.


In [ ]:
class PerturbedDomainDataset(
    DomainDataset
):
    def __init__(
        self,
        *args,
        perturbation_fn=None,
        **kwargs
    ):
        super().__init__(
            *args,
            **kwargs
        )

        self.perturbation_fn = (
            perturbation_fn
        )

    def __getitem__(
        self,
        index
    ):
        row = self.metadata.iloc[
            index
        ]

        image = self.images[
            int(
                row[
                    "tensor_index"
                ]
            )
        ].clone()

        if self.perturbation_fn is not None:
            image = self.perturbation_fn(
                image
            )

        if self.harmonization == "none":
            pass

        elif self.harmonization == "global":
            image = (
                image
                - self.train_mean
            ) / (
                self.train_std
                + 1e-8
            )

        elif self.harmonization == "per_image":
            image = (
                image
                - image.mean()
            ) / (
                image.std()
                + 1e-8
            )

        elif self.harmonization == "histogram_reference":
            image = histogram_match_tensor(
                image,
                self.histogram_reference
            )

        target = torch.tensor(
            int(
                row[
                    "label"
                ]
            ),
            dtype=torch.long
        )

        return (
            image,
            target,
            index
        )


# 53. Robustness Curve Helper


In [ ]:
def evaluate_perturbation_levels(
    model,
    metadata,
    images,
    levels,
    perturbation_builder,
    harmonization,
    train_mean,
    train_std,
    histogram_reference
):
    rows = []

    for level in levels:
        dataset = PerturbedDomainDataset(
            metadata,
            images,
            training=False,
            harmonization=harmonization,
            train_mean=train_mean,
            train_std=train_std,
            histogram_reference=(
                histogram_reference
            ),
            perturbation_fn=(
                perturbation_builder(
                    level
                )
            )
        )

        loader = DataLoader(
            dataset,
            batch_size=48,
            shuffle=False,
            num_workers=0
        )

        accuracy, _ = evaluate_model(
            model,
            dataset,
            loader,
            device
        )

        rows.append({
            "level":
                level,

            "accuracy":
                accuracy
        })

    return pd.DataFrame(
        rows
    )


# 54. Intensity Gain Stress Test


In [ ]:
global_model = harmonization_results[
    "global"
][
    "model"
]

gain_results = evaluate_perturbation_levels(
    global_model,
    val_metadata,
    domain_images,
    levels=[
        0.6,
        0.8,
        1.0,
        1.2,
        1.4
    ],
    perturbation_builder=lambda factor:
        (
            lambda image:
                perturb_gain(
                    image,
                    factor
                )
        ),
    harmonization="global",
    train_mean=train_mean,
    train_std=train_std,
    histogram_reference=(
        train_hist_reference
    )
)

print(
    gain_results
)


# 55. Plot Gain Robustness


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    gain_results[
        "level"
    ],
    gain_results[
        "accuracy"
    ],
    marker="o"
)

plt.xlabel(
    "Gain Factor"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Intensity Gain Robustness"
)

plt.show()


# 56. Noise Sensitivity


In [ ]:
noise_results = evaluate_perturbation_levels(
    global_model,
    val_metadata,
    domain_images,
    levels=[
        0.00,
        0.05,
        0.10,
        0.15,
        0.20
    ],
    perturbation_builder=lambda std:
        (
            lambda image:
                perturb_noise(
                    image,
                    std
                )
        ),
    harmonization="global",
    train_mean=train_mean,
    train_std=train_std,
    histogram_reference=(
        train_hist_reference
    )
)

print(
    noise_results
)


# 57. Plot Noise Robustness


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    noise_results[
        "level"
    ],
    noise_results[
        "accuracy"
    ],
    marker="o"
)

plt.xlabel(
    "Added Noise Standard Deviation"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Noise Sensitivity"
)

plt.show()


# 58. Resolution Sensitivity


In [ ]:
resolution_results = evaluate_perturbation_levels(
    global_model,
    val_metadata,
    domain_images,
    levels=[
        1.0,
        0.75,
        0.50,
        0.35,
        0.25
    ],
    perturbation_builder=lambda scale:
        (
            lambda image:
                perturb_resolution(
                    image,
                    scale
                )
        ),
    harmonization="global",
    train_mean=train_mean,
    train_std=train_std,
    histogram_reference=(
        train_hist_reference
    )
)

print(
    resolution_results
)


# 59. Robustness Is a Curve, Not One Number

A model can have:

$$
90\%
$$

clean-data accuracy but collapse under mild perturbation.

A robustness curve reveals how performance changes as the shift becomes stronger.


# 60. Stress Tests Should Be Realistic

Do not perturb images arbitrarily.

Stress tests should reflect plausible variation such as:

- Gain changes
- Depth/resolution changes
- Noise
- Blur
- Cropping
- Device post-processing

The test should model a deployment risk.


# 61. Harmonization Goal

A harmonizer should ideally reduce:

$$
Domain\ Information
$$

while preserving:

$$
Task\ Information
$$

This is the central tradeoff.


# 62. Feature-Space Harmonization Intuition

Instead of harmonizing raw pixels:

$$
x
$$

we can learn a representation:

$$
z=f(x)
$$

and try to make:

$$
P(z|Site_A)
\approx
P(z|Site_B)
$$

while still preserving label information.


# 63. Feature Extractor

We will expose CNN features before the classifier.


In [ ]:
class FeatureCNN(nn.Module):
    def __init__(
        self,
        num_classes=3
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                16,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(
                1
            )
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def extract_features(
        self,
        x
    ):
        x = self.features(
            x
        )

        return torch.flatten(
            x,
            start_dim=1
        )

    def forward(
        self,
        x
    ):
        features = self.extract_features(
            x
        )

        return self.classifier(
            features
        )


# 64. Domain Classifier Intuition

If site/device can be predicted accurately from learned features:

$$
z
$$

then those features contain domain information.

That may be harmless—or it may indicate shortcut risk.

A simple domain classifier can be used as a diagnostic.


# 65. Domain-Invariant Representation Learning

One family of methods tries to optimize:

$$
\boxed{
Good\ Label\ Prediction
}
$$

while simultaneously making:

$$
\boxed{
Poor\ Domain\ Prediction
}
$$

from the same features.

This encourages domain-invariant representations.


# 66. Gradient Reversal Intuition

Domain-adversarial training often uses:

> **Gradient Reversal Layer**

Forward pass:

$$
GRL(z)=z
$$

Backward pass:

$$
\frac{\partial GRL}{\partial z}
=
-\lambda I
$$

So the feature extractor learns to confuse the domain classifier.


In [ ]:
class GradientReversalFunction(
    torch.autograd.Function
):
    @staticmethod
    def forward(
        ctx,
        x,
        lambda_value
    ):
        ctx.lambda_value = (
            lambda_value
        )

        return x.view_as(
            x
        )

    @staticmethod
    def backward(
        ctx,
        grad_output
    ):
        return (
            -ctx.lambda_value
            * grad_output,
            None
        )


def gradient_reverse(
    x,
    lambda_value=1.0
):
    return (
        GradientReversalFunction
        .apply(
            x,
            lambda_value
        )
    )


# 67. Domain-Adversarial Model


In [ ]:
class DomainAdversarialCNN(
    nn.Module
):
    def __init__(
        self,
        num_classes=3,
        num_domains=3
    ):
        super().__init__()

        self.feature_extractor = (
            nn.Sequential(
                nn.Conv2d(
                    1,
                    16,
                    3,
                    padding=1
                ),
                nn.ReLU(),
                nn.MaxPool2d(2),

                nn.Conv2d(
                    16,
                    32,
                    3,
                    padding=1
                ),
                nn.ReLU(),

                nn.AdaptiveAvgPool2d(
                    1
                )
            )
        )

        self.label_head = nn.Linear(
            32,
            num_classes
        )

        self.domain_head = (
            nn.Sequential(
                nn.Linear(
                    32,
                    16
                ),
                nn.ReLU(),
                nn.Linear(
                    16,
                    num_domains
                )
            )
        )

    def extract_features(
        self,
        x
    ):
        x = self.feature_extractor(
            x
        )

        return torch.flatten(
            x,
            start_dim=1
        )

    def forward(
        self,
        x,
        lambda_domain=1.0
    ):
        features = self.extract_features(
            x
        )

        label_logits = (
            self.label_head(
                features
            )
        )

        reversed_features = (
            gradient_reverse(
                features,
                lambda_domain
            )
        )

        domain_logits = (
            self.domain_head(
                reversed_features
            )
        )

        return (
            label_logits,
            domain_logits
        )


# 68. Domain-Adversarial Loss

Training objective:

$$
\boxed{
L
=
L_{label}
+
L_{domain}
}
$$

because gradient reversal changes only the feature-extractor gradient from the domain branch.

The domain head itself still learns to classify domains.

The feature extractor learns to make that task harder.


# 69. Important Warning About Domain-Invariant Learning

Removing domain information is not always correct.

If domain correlates with:

- Real biological differences
- Population differences
- Disease severity

forcing perfect invariance may remove useful or necessary signal.

Domain invariance must be justified scientifically.


# 70. Detecting Shortcut Learning Across Sites

Suppose:

$$
Site_A
$$

contains mostly class 0 and:

$$
Site_C
$$

contains mostly class 2.

A model can use:

$$
Site\ Appearance
\rightarrow
Label
$$

instead of anatomy.

The first diagnostic is:

> Cross-tabulate site and label.


In [ ]:
print(
    pd.crosstab(
        internal_only[
            "site"
        ],
        internal_only[
            "label"
        ]
    )
)


# 71. Shortcut Stress Test

A strong diagnostic is:

> Evaluate on a domain where the site-label correlation changes.

If performance collapses, the model may have relied on that shortcut.


# 72. Synthetic Site Shortcut Marker

We can intentionally add a tiny site-specific marker.

This demonstrates how an easy nuisance cue can dominate learning.


In [ ]:
def add_site_marker(
    image,
    site
):
    image = image.clone()

    if site == "Site_A":
        image[
            :,
            1:5,
            1:5
        ] = 1.0

    elif site == "Site_B":
        image[
            :,
            1:5,
            -5:-1
        ] = 1.0

    elif site == "Site_C":
        image[
            :,
            -5:-1,
            1:5
        ] = 1.0

    return image


# 73. Why Site Markers Matter

Real analogues include:

- Scanner logos
- Depth-scale graphics
- Measurement calipers
- Text overlays
- Black border geometry
- Annotation conventions

Interpretability and counterfactual masking can help detect this.


# 74. Domain Prediction as a Shortcut Diagnostic

If a domain classifier can predict site with near-perfect accuracy from model features, the representation contains strong site information.

That does not automatically prove harmful shortcut use.

But it is a warning worth investigating.


# 75. Feature-Space Domain Distance

Instead of comparing raw images, compare:

$$
f(x)
$$

from the trained network.

A robust representation may show:

- Strong class separation
- Reduced site separation


# 76. Measuring Site Means in Feature Space

A simple diagnostic:

$$
\mu_{site}
=
\frac{1}{N_{site}}
\sum_i f(x_i)
$$

Then compare distances between site feature means.


In [ ]:
def feature_matrix_from_dataset(
    model,
    dataset,
    loader,
    device
):
    model.eval()

    features = []
    rows = []

    with torch.inference_mode():
        for images, targets, indices in loader:
            images = images.to(
                device
            )

            batch_features = (
                model.extract_features(
                    images
                )
                .cpu()
            )

            features.append(
                batch_features
            )

            for index in indices:
                local_index = int(
                    index.item()
                )

                rows.append(
                    dataset
                    .metadata
                    .iloc[
                        local_index
                    ]
                    .to_dict()
                )

    return (
        torch.cat(
            features
        ),
        pd.DataFrame(
            rows
        )
    )


# 77. Robustness Metrics Beyond Accuracy

For domain-shift studies, consider:

- Accuracy
- Macro F1
- Sensitivity/specificity
- AUROC/AUPRC
- Calibration
- Worst-site performance
- Worst-device performance
- Mean performance
- Performance variance across sites


# 78. Worst-Domain Performance

A model with:

$$
95\%
$$

average accuracy but:

$$
60\%
$$

on one scanner may be unacceptable.

A useful robustness metric is:

$$
\boxed{
\min_d Performance(d)
}
$$


In [ ]:
def subgroup_accuracy_table(
    prediction_df,
    column
):
    rows = []

    for subgroup, group in (
        prediction_df.groupby(
            column
        )
    ):
        accuracy = (
            group[
                "true_label"
            ].to_numpy()
            ==
            group[
                "predicted_label"
            ].to_numpy()
        ).mean()

        rows.append({
            column:
                subgroup,

            "n":
                len(
                    group
                ),

            "accuracy":
                float(
                    accuracy
                )
        })

    return pd.DataFrame(
        rows
    )


# 79. Validation Site Metrics


In [ ]:
global_val_predictions = (
    harmonization_results[
        "global"
    ][
        "predictions"
    ]
)

site_val_table = (
    subgroup_accuracy_table(
        global_val_predictions,
        "site"
    )
)

print(
    site_val_table
)

print(
    "Worst-site accuracy:",
    site_val_table[
        "accuracy"
    ].min()
)


# 80. Domain Performance Variance

Another robustness summary:

$$
Var(
Performance_d
)
$$

Lower variance can indicate more consistent domain behavior.

But consistency at uniformly poor performance is not useful.

Always report both level and variability.


In [ ]:
print(
    "Site accuracy std:",
    site_val_table[
        "accuracy"
    ].std()
)


# 81. Calibration Under Domain Shift

A model may remain accurate but become overconfident on a new scanner.

Therefore evaluate:

- Accuracy
- Calibration
- Confidence distribution

under domain shift.


# 82. Mean Confidence by Domain


In [ ]:
def add_confidence(
    prediction_df,
    num_classes=3
):
    probability_columns = [
        f"prob_class_{index}"
        for index in range(
            num_classes
        )
    ]

    dataframe = (
        prediction_df.copy()
    )

    dataframe[
        "confidence"
    ] = dataframe[
        probability_columns
    ].max(
        axis=1
    )

    return dataframe


confidence_df = add_confidence(
    global_val_predictions
)

print(
    confidence_df.groupby(
        "site"
    )[
        "confidence"
    ].mean()
)


# 83. Robustness Gap

A simple robustness gap is:

$$
\boxed{
Performance_{internal}
-
Performance_{external}
}
$$

Smaller is generally better, assuming both performances are high.


In [ ]:
global_internal_accuracy = (
    harmonization_results[
        "global"
    ][
        "val_accuracy"
    ]
)

global_external_accuracy = float(
    external_comparison[
        external_comparison[
            "harmonization"
        ]
        == "global"
    ][
        "external_accuracy"
    ].iloc[
        0
    ]
)

robustness_gap = (
    global_internal_accuracy
    - global_external_accuracy
)

print(
    "Robustness gap:",
    robustness_gap
)


# 84. Harmonization Evaluation Framework

A harmonization method should be evaluated on at least four dimensions:

$$
\begin{array}{|c|c|}
\hline
1 & \text{Task performance} \\
\hline
2 & \text{Unseen-domain generalization} \\
\hline
3 & \text{Domain-gap reduction} \\
\hline
4 & \text{Preservation of clinical signal} \\
\hline
\end{array}
$$


# 85. Domain-Gap Reduction Alone Is Not Enough

A method can make all sites look identical by destroying useful information.

Then domain distance decreases but classification also fails.

So:

$$
\boxed{
Lower\ Domain\ Distance
\not\Rightarrow
Better\ Model
}
$$


# 86. Performance Improvement Alone Is Not Enough

A harmonization method may accidentally amplify class-specific artifacts.

Then performance improves for the wrong reason.

So combine:

- Metrics
- Site/device analysis
- Stress tests
- Explanation checks


# 87. Compare Domain Histograms Before and After Harmonization

A useful diagnostic is to visualize:

- Raw site histograms
- Harmonized site histograms

But remember:

> Similar histograms do not prove equivalent image distributions.


In [ ]:
def collect_harmonized_values(
    metadata,
    images,
    mode,
    train_mean,
    train_std,
    histogram_reference,
    max_images=20
):
    values = []

    subset = metadata.iloc[
        :max_images
    ]

    for _, row in subset.iterrows():
        image = images[
            int(
                row[
                    "tensor_index"
                ]
            )
        ].clone()

        if mode == "global":
            image = (
                image
                - train_mean
            ) / (
                train_std
                + 1e-8
            )

        elif mode == "per_image":
            image = (
                image
                - image.mean()
            ) / (
                image.std()
                + 1e-8
            )

        elif mode == "histogram_reference":
            image = histogram_match_tensor(
                image,
                histogram_reference
            )

        values.append(
            image.flatten()
        )

    return torch.cat(
        values
    )


# 88. Mean/Std After Per-Image Standardization


In [ ]:
example_standardized = (
    (
        external_image
        - external_image.mean()
    )
    / (
        external_image.std()
        + 1e-8
    )
)

print(
    "Mean:",
    example_standardized.mean().item()
)

print(
    "Std:",
    example_standardized.std().item()
)


# 89. Per-Image Standardization Tradeoff

Advantages:

- Removes image-level brightness/gain
- Simple
- No device metadata required

Disadvantages:

- Removes absolute intensity scale
- Can amplify noise
- Can change clinically relevant contrast


# 90. Global Standardization Tradeoff

Advantages:

- Preserves relative image brightness
- Simple and stable
- Training-derived

Disadvantages:

- Device/site differences may remain


# 91. Histogram Matching Tradeoff

Advantages:

- Stronger global intensity alignment

Disadvantages:

- Can distort task-relevant intensities
- Reference-dependent
- Does not fix spatial/domain shifts


# 92. Feature-Space Harmonization Tradeoff

Advantages:

- Can target higher-level nuisance variation
- Can preserve raw image appearance

Disadvantages:

- Harder to train
- Can remove useful domain-correlated biology
- Requires careful validation


# 93. Domain-Invariant Representation Does Not Mean Domain-Blind Medicine

Some domain information may be clinically necessary.

For example, site may correlate with:

- Population
- Disease severity
- Protocol

The goal is not always:

$$
zero\ domain\ information
$$

The goal is:

> Remove nuisance information that harms intended generalization.


# 94. Evaluating Shortcut Learning With Counterfactuals

A useful test:

1. Identify suspicious region
2. Mask or modify it
3. Recompute prediction
4. Measure score change

If the model relies strongly on a scanner marker, prediction may shift sharply when the marker is removed.


# 95. Site/Device Labels Should Be Available for Analysis

Even if site/device are not inputs to the model, preserve them in metadata.

They are essential for:

- Subgroup evaluation
- Domain shift analysis
- Harmonization studies
- Shortcut diagnostics


# 96. Harmonization Should Be Fit Inside Every CV Fold

For patient-grouped cross-validation:

For fold $k$:

$$
Train_k
\rightarrow
Fit\ Harmonizer_k
$$

Then:

$$
Harmonizer_k(
Validation_k
)
$$

Do not fit one harmonizer on all development patients.


# 97. External Test Must Stay External

If you choose the harmonization method by external-site accuracy, that site becomes part of development.

Use internal validation/CV for selection.

Use external site only for final evidence.


# 98. Robustness Stress Tests Should Also Be Validation-Only During Development

If you repeatedly choose the model that performs best on one fixed stress test, that stress test becomes another development target.

Define stress tests before final evaluation when possible.


# 99. Robustness Is Multi-Dimensional

A robust ultrasound model should ideally remain stable across:

- Sites
- Devices
- Gain changes
- Noise changes
- Resolution changes
- Mild intensity shifts
- Reasonable view variation


# 100. Repeated Seeds for Robustness Studies

Domain robustness comparisons should be repeated across the same seed set.

Otherwise one method may look better because of training randomness.


# 101. Patient-Grouped CV for Harmonization

A strong harmonization study may use:

$$
5\ folds
\times
3\ seeds
\times
4\ harmonization\ methods
$$

which already gives:

$$
60
$$

training runs.

Plan compute carefully.


# 102. Site-Held-Out vs Patient-Grouped CV

Patient-grouped CV answers:

> Does the model generalize to unseen patients from familiar domains?

Site-held-out answers:

> Does the model generalize to a new site?

Both are valuable and answer different questions.


# 103. Device-Held-Out vs External Site

Device-held-out isolates hardware/generalization better.

External-site testing may combine:

- Device
- Population
- Workflow
- Operator
- Labeling differences

So external validation is broader but less controlled.


# 104. Distribution Metrics Should Be Label-Aware When Needed

Suppose Site A contains mostly class 0 and Site B mostly class 2.

A raw distribution distance may reflect class composition rather than scanner shift.

Better analyses may compare domains **within class**.


# 105. Class-Conditional Domain Comparison

For class $c$ compare:

$$
P(X|Site_A,Y=c)
$$

with:

$$
P(X|Site_B,Y=c)
$$

This reduces confounding by label prevalence.


In [ ]:
def class_conditional_mean(
    metadata,
    images,
    site,
    class_index
):
    subset = metadata[
        (
            metadata[
                "site"
            ]
            == site
        )
        &
        (
            metadata[
                "label"
            ]
            == class_index
        )
    ]

    if len(
        subset
    ) == 0:
        return float(
            "nan"
        )

    stacked = torch.stack([
        images[
            int(
                index
            )
        ]
        for index in subset[
            "tensor_index"
        ].tolist()
    ])

    return float(
        stacked.mean().item()
    )


for class_index in range(3):
    print(
        "Class",
        class_index,
        "Site A mean:",
        class_conditional_mean(
            internal_only,
            domain_images,
            "Site_A",
            class_index
        )
    )


# 106. Label Shift Detection

A basic label-shift diagnostic is simply to compare class prevalence:

$$
P(Y)
$$

across sites.


In [ ]:
site_prevalence = pd.crosstab(
    domain_metadata[
        "site"
    ],
    domain_metadata[
        "label"
    ],
    normalize="index"
)

print(
    site_prevalence
)


# 107. Precision Changes Under Label Shift

Even if sensitivity/specificity stay constant, precision can change when prevalence changes.

This is why deployment prevalence matters.


# 108. Calibration Can Shift Across Sites

A probability:

$$
0.8
$$

may mean different observed risk in different domains if calibration drifts.

For clinical deployment, site-specific calibration checks may be needed.


# 109. Domain Shift Can Affect Threshold Choice

A threshold selected on one site may not preserve:

- Sensitivity
- Specificity
- Precision

on another site.

Therefore evaluate the operating point on external data.


# 110. Harmonization Can Change Calibration

Even when AUROC improves, probability calibration can worsen.

Always evaluate the metric that matters for the final task.


# 111. Common Mistake — Harmonizing Before Splitting

If harmonization uses all data before split creation, test information leaks into the pipeline.


# 112. Common Mistake — Fitting Histogram Reference on External Data

The external distribution must not define the training transformation unless that is explicitly part of the deployment protocol.


# 113. Common Mistake — Using Per-Device Normalization With Test Device Statistics

If the unseen device provides its own mean/std during test-time evaluation, you may accidentally create transductive leakage.

Be explicit about what information is available at deployment.


# 114. Common Mistake — Assuming Lower MMD Means Better Harmonization

MMD can decrease while label information disappears.

Always connect domain distance to downstream task performance.


# 115. Common Mistake — Evaluating Only Average Accuracy

Report:

- Mean performance
- Worst-site performance
- Worst-device performance
- External performance
- Variability


# 116. Common Mistake — Stress Testing With Unrealistic Perturbations

A robustness test should represent plausible ultrasound variation.

Otherwise the result may not be clinically meaningful.


# 117. Common Mistake — Treating Histogram Matching as a Universal Solution

Histogram matching only aligns global intensity distribution.

It does not guarantee:

- Feature alignment
- Scanner invariance
- Robust generalization


# 118. Common Mistake — Removing Every Domain Signal

Some domain differences may reflect real biological/population differences.

Blindly removing all domain information can create bias.


# 119. Common Mistake — Using Site as a Hidden Label Shortcut

Always inspect:

```python
pd.crosstab(
    metadata["site"],
    metadata["label"]
)
```

and the same for device.


# 120. Common Mistake — Choosing Harmonization on the Test Set

Harmonization is part of model development.

Choose it on validation/CV.

Evaluate final external test once.


# 121. Common Mistake — Ignoring Domain-Specific Failure Cases

Inspect errors by:

- Site
- Device
- Class
- Image quality
- Gain/intensity range
- Resolution


# 122. A Research-Grade Harmonization Protocol

A strong protocol can be:

1. Patient-grouped folds
2. Training-only harmonization fit
3. Same model/seed/folds for every harmonization method
4. Internal CV
5. Site/device-held-out tests
6. Stress tests
7. External-site test
8. Patient-level confidence intervals


# 123. What Should Be Reported?

For every harmonization method, consider reporting:

- Internal CV mean
- Internal CV variability
- Worst-site score
- Worst-device score
- Site-held-out score
- Device-held-out score
- External-site score
- Robustness curves
- Calibration


# 124. Harmonization Ablation Table

A useful paper table might look like:

$$
\begin{array}{|c|c|c|c|c|}
\hline
Method &
Internal &
Worst\ Site &
External &
Robustness\ Gap \\
\hline
None & \cdots & \cdots & \cdots & \cdots \\
\hline
Global & \cdots & \cdots & \cdots & \cdots \\
\hline
Per\ Image & \cdots & \cdots & \cdots & \cdots \\
\hline
Histogram & \cdots & \cdots & \cdots & \cdots \\
\hline
\end{array}
$$


# 125. Robustness Result Table From Our Demo


In [ ]:
robustness_table = (
    external_comparison.copy()
)

robustness_table[
    "robustness_gap"
] = (
    robustness_table[
        "internal_val_accuracy"
    ]
    -
    robustness_table[
        "external_accuracy"
    ]
)

print(
    robustness_table
)


# 126. Saving Robustness Outputs

For reproducibility, save:

- Harmonization configs
- Site/device split definitions
- Stress-test definitions
- Prediction tables
- Robustness curves
- External results


In [ ]:
ROBUSTNESS_DIR = Path(
    "robustness_results"
)

ROBUSTNESS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

robustness_table.to_csv(
    ROBUSTNESS_DIR
    / "harmonization_comparison.csv",
    index=False
)

site_holdout_df.to_csv(
    ROBUSTNESS_DIR
    / "site_holdout.csv",
    index=False
)

device_holdout_df.to_csv(
    ROBUSTNESS_DIR
    / "device_holdout.csv",
    index=False
)

gain_results.to_csv(
    ROBUSTNESS_DIR
    / "gain_stress_test.csv",
    index=False
)

noise_results.to_csv(
    ROBUSTNESS_DIR
    / "noise_stress_test.csv",
    index=False
)

resolution_results.to_csv(
    ROBUSTNESS_DIR
    / "resolution_stress_test.csv",
    index=False
)

print(
    "Robustness outputs saved."
)


# 127. Practice Exercises

## Exercise 1

Explain the difference between:

- Covariate shift
- Label shift
- Concept shift

## Exercise 2

Compute mean intensity and standard deviation for each site.

## Exercise 3

Plot site-specific intensity histograms.

## Exercise 4

Create a site-held-out train/test split.

## Exercise 5

Create a device-held-out train/test split.

## Exercise 6

Implement per-image standardization.

## Exercise 7

Implement histogram matching using a training-derived reference.

## Exercise 8

Run a gain robustness stress test.

## Exercise 9

Run a resolution-degradation stress test.

## Exercise 10

Compute the robustness gap between internal and external accuracy.


# 128. Conceptual Challenges

## Challenge 1

Why can a model fail when the scanner changes even if the disease definition is unchanged?

## Challenge 2

What is covariate shift?

## Challenge 3

What is label shift?

## Challenge 4

Why can precision change under label shift?

## Challenge 5

Why is site-held-out evaluation different from patient-grouped CV?

## Challenge 6

Why is device-held-out evaluation useful?

## Challenge 7

Why must harmonization be fitted using training data only?

## Challenge 8

Why can per-image standardization improve robustness?

## Challenge 9

Why can per-image standardization remove useful clinical information?

## Challenge 10

Why does histogram matching not solve every form of scanner shift?

## Challenge 11

Why does lower domain distance not necessarily mean better classification?

## Challenge 12

What is domain-invariant representation learning?

## Challenge 13

Why can forcing complete domain invariance be harmful?

## Challenge 14

How can scanner/site shortcuts enter ultrasound models?

## Challenge 15

What evidence would convince you that a harmonization method truly improved generalization?


# 129. Exercise Solutions


In [ ]:
# Exercise 2
exercise_site_stats = []

for site, group in (
    domain_metadata.groupby(
        "site"
    )
):
    stacked = torch.stack([
        domain_images[
            int(
                index
            )
        ]
        for index in group[
            "tensor_index"
        ].tolist()
    ])

    exercise_site_stats.append({
        "site":
            site,

        "mean":
            float(
                stacked.mean().item()
            ),

        "std":
            float(
                stacked.std().item()
            )
    })

exercise_site_stats = pd.DataFrame(
    exercise_site_stats
)

print(
    exercise_site_stats
)


In [ ]:
# Exercise 4
exercise_train_site, exercise_test_site = (
    site_held_out_split(
        internal_only,
        "Site_C"
    )
)

print(
    exercise_train_site[
        "site"
    ].unique()
)

print(
    exercise_test_site[
        "site"
    ].unique()
)


In [ ]:
# Exercise 5
exercise_train_device, exercise_test_device = (
    device_held_out_split(
        internal_only,
        "Device_C"
    )
)

print(
    exercise_train_device[
        "device"
    ].unique()
)

print(
    exercise_test_device[
        "device"
    ].unique()
)


In [ ]:
# Exercise 6
exercise_image = domain_images[
    0
]

exercise_standardized = (
    exercise_image
    - exercise_image.mean()
) / (
    exercise_image.std()
    + 1e-8
)

print(
    exercise_standardized.mean().item(),
    exercise_standardized.std().item()
)


In [ ]:
# Exercise 7
exercise_reference = (
    build_histogram_reference(
        train_metadata,
        domain_images
    )
)

exercise_matched = (
    histogram_match_tensor(
        external_image,
        exercise_reference
    )
)

print(
    exercise_matched.shape
)


In [ ]:
# Exercise 8
exercise_gain_results = (
    evaluate_perturbation_levels(
        global_model,
        val_metadata,
        domain_images,
        levels=[
            0.8,
            1.0,
            1.2
        ],
        perturbation_builder=lambda factor:
            (
                lambda image:
                    perturb_gain(
                        image,
                        factor
                    )
            ),
        harmonization="global",
        train_mean=train_mean,
        train_std=train_std,
        histogram_reference=(
            train_hist_reference
        )
    )
)

print(
    exercise_gain_results
)


In [ ]:
# Exercise 9
exercise_resolution_results = (
    evaluate_perturbation_levels(
        global_model,
        val_metadata,
        domain_images,
        levels=[
            1.0,
            0.5,
            0.25
        ],
        perturbation_builder=lambda scale:
            (
                lambda image:
                    perturb_resolution(
                        image,
                        scale
                    )
            ),
        harmonization="global",
        train_mean=train_mean,
        train_std=train_std,
        histogram_reference=(
            train_hist_reference
        )
    )
)

print(
    exercise_resolution_results
)


In [ ]:
# Exercise 10
exercise_gap = (
    global_internal_accuracy
    - global_external_accuracy
)

print(
    "Robustness gap:",
    exercise_gap
)


# 130. Conceptual Challenge Solutions

## Challenge 1

The scanner changes the image distribution. Even when the clinical label meaning stays the same, the model may have learned scanner-dependent visual statistics.

## Challenge 2

Covariate shift means the input distribution changes:

$$
P_{train}(X)
\neq
P_{test}(X)
$$

while the task relationship is assumed to remain similar.

## Challenge 3

Label shift means class prevalence changes:

$$
P_{train}(Y)
\neq
P_{test}(Y)
$$

## Challenge 4

Precision depends on prevalence. Even with unchanged sensitivity and specificity, a rarer positive class usually lowers positive predictive value.

## Challenge 5

Patient-grouped CV tests new patients from familiar development domains. Site-held-out evaluation tests an unseen acquisition site.

## Challenge 6

It directly measures robustness to scanner/device-specific acquisition differences.

## Challenge 7

Otherwise validation/test information influences the transformation and leaks into model development.

## Challenge 8

It removes image-level brightness and scale differences that may come from gain or scanner processing.

## Challenge 9

Absolute echogenicity or intensity relationships may themselves contain clinically meaningful information.

## Challenge 10

It aligns global intensity distributions but cannot fix differences in resolution, texture, geometry, artifacts, view, or population.

## Challenge 11

A harmonizer can reduce site differences by destroying both nuisance and label information.

## Challenge 12

It is the idea of learning features that retain task information while reducing domain/site/device information.

## Challenge 13

Domain information can sometimes correlate with real biological or population differences. Removing all of it may remove useful signal.

## Challenge 14

Through logos, markers, borders, gain patterns, post-processing, device-specific noise, or site-specific acquisition protocols.

## Challenge 15

Evidence should include improved unseen-site/device performance, preserved class performance, stable calibration, robustness under realistic perturbations, and no new shortcut behavior.


# 131. Key Takeaways

In this notebook, we studied:

- Domain shift
- Scanner shift
- Site shift
- Covariate shift
- Label shift
- Concept shift
- Distribution statistics
- Intensity histograms
- MMD intuition
- Site-held-out evaluation
- Device-held-out evaluation
- Gain stress tests
- Noise stress tests
- Resolution stress tests
- Robustness curves
- Global standardization
- Per-image standardization
- Histogram matching
- Feature-space harmonization
- Domain-adversarial learning
- Gradient reversal
- Shortcut learning
- Domain classifiers
- Worst-site performance
- Robustness gaps
- Calibration under shift
- Class-conditional distribution comparisons
- Leakage-safe harmonization
- External validation

The most important domain-shift equation is:

$$
\boxed{
P_{train}(X,Y)
\neq
P_{test}(X,Y)
}
$$

The most important harmonization principle is:

$$
\boxed{
Reduce\ Nuisance
\quad
while\ preserving\ Task\ Signal
}
$$

And the most important evaluation principle is:

$$
\boxed{
Harmonization\ Success
=
Better\ Unseen\ Domain\ Generalization
}
$$

—not simply more visually similar images.


# 132. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is domain shift?
2. What causes scanner shift in ultrasound?
3. What is site shift?
4. What is covariate shift?
5. What is label shift?
6. What is concept shift?
7. Why can site and scanner shift occur simultaneously?
8. Why inspect intensity distributions across sites?
9. What does MMD try to measure?
10. Why does distribution distance not directly equal performance loss?
11. What is site-held-out evaluation?
12. What is device-held-out evaluation?
13. What is a robustness stress test?
14. Why test gain sensitivity?
15. Why test noise sensitivity?
16. Why test resolution sensitivity?
17. What is global standardization?
18. What is per-image standardization?
19. Why can per-image normalization help?
20. Why can it hurt?
21. What is histogram matching?
22. Why must its reference come from training data?
23. What is feature-space harmonization?
24. What is domain-invariant representation learning?
25. What does gradient reversal do?
26. Why can complete domain invariance be harmful?
27. What is shortcut learning?
28. How can site/device become a shortcut?
29. What is worst-site performance?
30. What evidence should demonstrate that harmonization truly improved robustness?


# Next Notebook

# 27 — Multimodal Learning: Combining Ultrasound Images and Clinical Features

In the next notebook, we will study:

- Why combine imaging and tabular data?
- Clinical feature preprocessing
- Missing-value handling
- Numerical and categorical variables
- Image encoder
- Tabular encoder
- Feature fusion
- Early vs late fusion
- Concatenation-based multimodal networks
- Training multimodal models
- Preventing leakage from clinical variables
- Modality ablation studies
- Comparing image-only vs tabular-only vs fused models
- Patient-level multimodal evaluation
- Handling missing modalities
- Preparing a research-quality multimodal ultrasound pipeline
